In [25]:

from langgraph.graph import StateGraph , START, END
from langchain_groq import ChatGroq
from typing import List, Dict, Any, TypedDict
from dotenv import load_dotenv
import os


In [26]:
load_dotenv()  # Load environment variables from .env file

model = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.environ.get("GROQ_API_KEY")
)

In [27]:
 #defined the state
class PromptChaining(TypedDict):
    topic: str
    outline: str
    blog: str
    

In [28]:
# node 1
def outline_generator(state: PromptChaining) -> PromptChaining:
    topic = state['topic']
    prompt = f"Write an outline for a blog post about {topic}"
    
    response = model.invoke(prompt)
    
    outline = response.content
    
    state['outline'] = outline
    
    return state


def blog_generator(state: PromptChaining) -> PromptChaining:
    outline = state['outline']
    
    prompt = f"Write a blog post based on the following outline: {outline}"
    
    response = model.invoke(prompt)
    
    blog = response.content
    
    state['blog'] = blog
    
    return state

    
    
    
  

    

In [29]:
# create a graph 
graph = StateGraph(PromptChaining)


# craate a nodes
graph.add_node('outline', outline_generator)
graph.add_node('blog', blog_generator)


# create a edges

graph.add_edge(START, 'outline')
graph.add_edge('outline', 'blog')
graph.add_edge('blog', END)


# compile the graph

graph.compile()
workflow = graph.compile()



In [30]:
initial_state = {'topic': 'write a blog on how coding works'}

result = workflow.invoke(initial_state)

print(result)

{'topic': 'write a blog on how coding works', 'outline': '**Title (suggested):** *“Demystifying the Magic: How Coding Actually Works – A Beginner’s Guide”*  \n\n---\n\n## 1. Introduction  \n- **Hook:** A relatable anecdote or statistic (e.g., “Every time you swipe right, a line of code makes it happen”).  \n- **Why it matters:** Understanding the fundamentals helps you learn faster, debug better, and appreciate technology.  \n- **What readers will get:** A clear, step‑by‑step walkthrough of what coding is, how code becomes an app, and where to go next.\n\n---\n\n## 2. What Is “Coding” Anyway?  \n- **Definition:** Translating human‑readable instructions into a language a computer can execute.  \n- **Key terms:**  \n  - *Source code*  \n  - *Programming language*  \n  - *Syntax vs. semantics*  \n- **Analogy:** Writing a recipe vs. a chef preparing the dish.\n\n---\n\n## 3. The Building Blocks of Code  \n### 3.1 Variables & Data Types  \n- Numbers, strings, booleans, arrays, objects.  \n-

In [31]:
print(result['blog'])

# Demystifying the Magic: How Coding Actually Works – A Beginner’s Guide  

*Ever wondered what’s really happening behind that “swipe right” that matches you on a dating app? A line of code is doing the heavy lifting. In this post we’ll pull back the curtain, walk through the journey from a simple idea to a running program, and give you a roadmap for your own coding adventures.*

---  

## 1. Introduction  

### Hook  
Picture this: you’re scrolling through your phone, you tap a button, and instantly a list of restaurants appears. No one is manually typing those results into your screen—**a handful of lines of code just did it**.  

### Why it matters  
Understanding the fundamentals of coding does three things:  

1. **Learning faster** – When you know what each piece does, you can pick up new languages with less friction.  
2. **Debugging smarter** – You’ll spot the difference between a typo and a logic error in seconds.  
3. **Appreciating the tech** – Knowing the “why” behind the “